In [11]:
import pandas as pd
import joblib
import os

# Ensure we load from ml-service directory
load_path = "network_congestion.csv"
if not os.getcwd().endswith("ml-service") and os.path.exists("netpulseai/ml-service"):
    load_path = os.path.join("netpulseai", "ml-service", "network_congestion.csv")

df = pd.read_csv(load_path)

df = df.sort_values("timestamp")

# Add rolling features
df["users_roll"] = df["active_users"].rolling(window=5).mean()
df["latency_roll"] = df["latency"].rolling(window=5).mean()
df["throughput_roll"] = df["throughput"].rolling(window=5).mean()

# Lag features
df["latency_lag1"] = df["latency"].shift(1)
df["throughput_lag1"] = df["throughput"].shift(1)
df["users_lag1"] = df["active_users"].shift(1)

# Rolling std deviation
df["latency_std"] = df["latency"].rolling(5).std()
df["throughput_std"] = df["throughput"].rolling(5).std()

# Rate of change
df["latency_change"] = df["latency"].pct_change()
df["throughput_change"] = df["throughput"].pct_change()

# Remove NaN values (first few rows)
df = df.dropna()

# Drop timestamp (or extract features from it)
df = df.drop(columns=["timestamp"])

# Features & target
X = df.drop(columns=["congestion_level"])
y = df["congestion_level"]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
print(y_train.value_counts())

congestion_level
0    7494
1    5595
2    2907
Name: count, dtype: int64


In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=15,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
joblib.dump(model, "model.pkl")

['model.pkl']

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.893

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.92      0.94      1874
           1       0.83      0.88      0.85      1399
           2       0.88      0.84      0.86       727

    accuracy                           0.89      4000
   macro avg       0.89      0.88      0.88      4000
weighted avg       0.90      0.89      0.89      4000


Confusion Matrix:
 [[1732  142    0]
 [  87 1232   80]
 [   0  119  608]]
